In [12]:
import unicodedata
from pathlib import Path

import pandas as pd
import numpy as np
from scipy.stats import spearmanr

In [13]:
TARGET_WORDS = [
    "überspannen",
    "Manschette",
    "Fuß",
    "Rezeption",
    "abgebrüht",
    "Dynamik",
    "Engpaß",
    "abbauen",
    "Mißklang",
    "Abgesang",
    "Knotenpunkt",
    "Spielball",
    "zersetzen",
    "Armenhaus",
    "Ohrwurm",
    "Eintagsfliege",
    "Seminar",
    "Sensation",
    "Titel",
    "Schmiere",
    "ausspannen",
    "packen",
    "artikulieren",
    "abdecken",
]

In [14]:
annotations_schemas = {
    "en-en": "schemas_for_german/en-en/german",
    "ru-ru": "schemas_for_german/ru-ru/german",
    "rusemshift-finetune": "schemas_for_german/rusemshift/finetune/german",
    "rusemshift-train": "schemas_for_german/rusemshift/train/german",
}

In [15]:
def decode_hash_unicode(name: str) -> str:
    return name.replace("u#U0308", "ü").replace("#U00df", "ß")

In [ ]:
def load_scores_for_word(
    schema_base_path: str,
    word: str,
    pairs_base_path: str,
    senses: pd.DataFrame,
) -> pd.DataFrame:

    base = Path(schema_base_path)
    word_nfc = unicodedata.normalize("NFC", word)

    word_dir = None
    for subdir in base.iterdir():
        if subdir.is_dir():
            decoded = unicodedata.normalize("NFC", decode_hash_unicode(subdir.name))
            if decoded == word_nfc:
                word_dir = subdir
                break

    if word_dir is None:
        print(f"  Directory not found for: {word}")
        return None

    score_files = list(word_dir.glob("*.scores"))
    if not score_files:
        print(f"  No scores file in: {word_dir}")
        return None

    scores_data = pd.read_json(score_files[0])
    scores_data["score"] = scores_data["score"].apply(
        lambda x: np.mean([float(v) for v in x])
    )

    word_encoded = word_dir.name
    pair_file = Path(pairs_base_path) / word_encoded / f"dev.{word_encoded}.data"
    if not pair_file.exists():
        print(f"  Pair file not found for: {word}")
        return None

    pairs = pd.read_json(pair_file)
    merged = pd.merge(pairs, scores_data, on="id")
    
    context_to_grouping = dict(zip(senses["context"], senses["grouping"]))
    merged["grouping1"] = merged["sentence1"].map(context_to_grouping)
    merged["grouping2"] = merged["sentence2"].map(context_to_grouping)
    
    cross = merged[merged["grouping1"] != merged["grouping2"]]
    print(f"  {word}: total={len(merged)} cross-perid={len(cross)}")
    
    if len(cross) == 0:
        print(f"  No cross-period pairs for: {word}")
        return None
    
    return cross[["score"]]
    
    

In [19]:
def compute_apd(scores_df: pd.DataFrame) -> float:
    if scores_df is None or len(scores_df) == 0:
        return None

    return float((1 - scores_df["score"]).mean())

In [ ]:
senses = pd.read_csv(
    "summer-wsi/datasets/se20lscd_v2/de/sense-old+new.tsv",
    sep="t",
)
pairs_base_path = "Serge/german"
print(f"Senses loaded: {senses.shape}")
print(senses[["context", "grouping"]].head())

In [ ]:
results = {}

for schema_name, schema_path in annotations_schemas.items():
    print(f"\n=== {schema_name} ===")
    results[schema_name] = {}

    for word in TARGET_WORDS:
        scores_df = load_scores_for_word(
            schema_path,
            word,
            pairs_base_path,
            senses,
        )
        apd = compute_apd(scores_df)
        results[schema_name][word] = apd
        if apd is not None:
            print(f"  {word}: APD = {apd:.4f}")
        else:
            print(f"  {word}: no data")


=== en-en ===
  überspannen: APD = 0.5086
  Manschette: APD = 0.4903
  Fuß: APD = 0.5183
  Rezeption: APD = 0.5130
  abgebrüht: APD = 0.5033
  Dynamik: APD = 0.5089
  Engpaß: APD = 0.5044
  abbauen: APD = 0.5086
  Mißklang: APD = 0.4872
  Abgesang: APD = 0.4891
  Knotenpunkt: APD = 0.5032
  Spielball: APD = 0.4860
  zersetzen: APD = 0.5046
  Armenhaus: APD = 0.4953
  Ohrwurm: APD = 0.4944
  Eintagsfliege: APD = 0.4848
  Seminar: APD = 0.4961
  Sensation: APD = 0.4958
  Titel: APD = 0.5081
  Schmiere: APD = 0.5178
  ausspannen: APD = 0.5176
  packen: APD = 0.5184
  artikulieren: APD = 0.5064
  abdecken: APD = 0.5121

=== ru-ru ===
  überspannen: APD = 0.5054
  Manschette: APD = 0.4909
  Fuß: APD = 0.5088
  Rezeption: APD = 0.5033
  abgebrüht: APD = 0.4989
  Dynamik: APD = 0.4993
  Engpaß: APD = 0.4991
  abbauen: APD = 0.5049
  Mißklang: APD = 0.4962
  Abgesang: APD = 0.4946
  Knotenpunkt: APD = 0.4988
  Spielball: APD = 0.4922
  zersetzen: APD = 0.5020
  Armenhaus: APD = 0.4935
  Ohrwu

In [21]:
def load_gold_data(path: str) -> dict:
    df = pd.read_csv(path, sep="\t")
    try:
        return dict(zip(df["lemma"], df["change_graded"]))
    except KeyError:
        return dict(zip(df["word"], df["change_graded"]))


gold_data = load_gold_data("gold-data-de.csv")

print("Spearman correlations: ")
for schema_name in annotations_schemas:
    pairs = [
        (results[schema_name][w], gold_data[w])
        for w in TARGET_WORDS
        if results[schema_name].get(w) is not None and w in gold_data
    ]
    if len(pairs) < 2:
        print(f"  {schema_name}: not enough data")
        continue

    apd_values, gold_values = zip(*pairs)
    spearman, pvalue = spearmanr(gold_values, apd_values)
    print(f"  {schema_name}: Spearman = {spearman:.4f} (p = {pvalue:.4f})")

Spearman correlations: 
  en-en: Spearman = 0.0043 (p = 0.9839)
  ru-ru: Spearman = 0.0365 (p = 0.8655)
  rusemshift-finetune: Spearman = 0.1696 (p = 0.4283)
  rusemshift-train: Spearman = 0.0678 (p = 0.7528)


In [22]:
rows = []
for schema_name in annotations_schemas:
    for word in TARGET_WORDS:
        rows.append(
            {
                "schema": schema_name,
                "word": word,
                "apd": results[schema_name].get(word),
            }
        )

summary_df = pd.DataFrame(rows)
summary_df.pivot(index="word", columns="schema", values="apd").round(4)

schema,en-en,ru-ru,rusemshift-finetune,rusemshift-train
word,,,,
Abgesang,0.4891,0.4946,0.4972,0.4980
Armenhaus,0.4953,0.4935,0.5033,0.4944
Dynamik,0.5089,0.4993,0.5045,0.5035
Eintagsfliege,0.4848,0.4901,0.4937,0.4924
Engpaß,0.5044,0.4991,0.5063,0.5036
Fuß,0.5183,0.5088,0.5142,0.5121
Knotenpunkt,0.5032,0.4988,0.5075,0.5005
Manschette,0.4903,0.4909,0.4900,0.4950
Mißklang,0.4872,0.4962,0.4966,0.5000
